In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pathlib import Path
import os

class GPTChatClient:
    """Loads GPT config from .env, wraps chat(), and prettifies outputs."""
    def __init__(
        self,
        system_prompt: str | None = None,
        temperature: float = 0,
        render_markdown: bool = True,
        title: str | None = "GPT-5.2 Response",
        max_fasta_chars: int = 6000,
    ):
        load_dotenv()
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise RuntimeError("请先在 .env 中设置 OPENROUTER_API_KEY")
        base_url = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")
        model = os.getenv("OPENROUTER_MODEL", "openai/gpt-5.2:online")

        self._client = OpenAI(api_key=api_key, base_url=base_url)
        self._model = model
        self._system_prompt = system_prompt
        self._default_temperature = temperature
        self._render_markdown = render_markdown
        self._markdown_title = title
        self._max_fasta_chars = max_fasta_chars

    def chat(
        self,
        prompt: str | list[dict],
        temperature: float | None = None,
        fasta_file: str | Path | None = None,
    ) -> str:
        """Send a prompt, optionally embed parsed FASTA text, and return GPT output."""
        messages: list[dict] = []
        if self._system_prompt:
            messages.append({"role": "system", "content": self._system_prompt})

        fasta_payload = None
        if fasta_file:
            fasta_payload = self._load_fasta_text(fasta_file)

        if isinstance(prompt, str):
            final_prompt = prompt
            if fasta_payload:
                final_prompt = self._attach_fasta_to_prompt(prompt, fasta_payload)
            messages.append({"role": "user", "content": final_prompt})
        else:
            messages.extend(prompt)
            if fasta_payload:
                messages.append({"role": "user", "content": self._format_fasta_block(fasta_payload)})

        resp = self._client.chat.completions.create(
            model=self._model,
            messages=messages,
            temperature=self._default_temperature if temperature is None else temperature,
        )
        content = resp.choices[0].message.content or ""
        if self._render_markdown:
            self._display_markdown(content)
        else:
            print(content)
        return content

    def _load_fasta_text(self, fasta_file: str | Path) -> dict:
        path = Path(fasta_file)
        if not path.exists():
            raise FileNotFoundError(f"未找到 FASTA 文件: {path}")
        text = path.read_text(encoding="utf-8", errors="ignore").strip()
        truncated = len(text) > self._max_fasta_chars
        if truncated:
            text = text[: self._max_fasta_chars]
        return {
            "filename": path.name,
            "content": text,
            "truncated": truncated,
        }

    def _attach_fasta_to_prompt(self, prompt: str, payload: dict) -> str:
        block = self._format_fasta_block(payload)
        return f"{prompt.strip()} {block}"

    def _format_fasta_block(self, payload: dict) -> str:
        truncated_flag = " (content truncated)" if payload["truncated"] else ""
        return f"[The content of the file after parsing is:：[{payload['content']}{truncated_flag}]]"

    def _display_markdown(self, content: str) -> None:
        body = (content.strip() or "_No content returned._").replace("\n", "\n\n")
        title = f"### {self._markdown_title}\n\n" if self._markdown_title else ""
        display(Markdown(f"{title}{body}"))

In [2]:
system_prompt = '''
You are EnzymeAnnotator. Your ONLY tool is web browsing. The user may give a UniProt accession and ask specific questions. Do: fetch → extract → answer ONLY what they asked, with citations. Never invent.

Steps
1) Open the exact UniProt entry for the accession (search: "<ACC> site:uniprot.org") and confirm it matches.
2) Pull only the fields needed to answer the user’s request (enzyme? EC? active-site residues? etc.).
3) If UniProt lacks the requested detail, optionally check RCSB PDB / InterPro / M-CSA(=CSA) to support that specific item.

Rules
- Every factual claim needs a URL citation (prefer UniProt).
- Mark curated/experimental vs predicted/inferred; if missing, say “not found in authoritative sources”.
- If the accession is invalid/ambiguous, say so and ask for the minimum clarification.

'''

In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound NC1=C2N=CN([C@@H]3O[C@H](CO)[C@@H](O)[C@H]3O)C2=NC=N1 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/1.fasta')

### GPT-5.2 Response

Yes — that SMILES is **adenosine** (specifically **9-β-D-ribofuranosyladenine**), a **purine nucleoside** made of **adenine + ribose** 🧬

A few quick identifiers:

* Molecular formula: **C₁₀H₁₃N₅O₄**
* Molecular weight: **~267.24 g/mol**
* What it is: one of the canonical nucleosides in **RNA**, and the core nucleoside unit found in **ATP/ADP/AMP** and many cofactors.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound [NH3+][C@@H](CCC(=O)N[C@@H](CS)C(=O)NCC(=O)[O-])C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/2.fasta')


### GPT-5.2 Response

That SMILES is **reduced glutathione (GSH)**: γ-Glu-Cys-Gly (a tripeptide with a free thiol on cysteine) 🙂.

For the enzyme in your uploaded **2.fasta**: based on a quick sequence sanity-check (length ≈ **609 aa**, no obvious N-terminal signal peptide, and it *doesn’t* show the classic signatures you’d expect for the most common “GSH-as-substrate” enzyme families like **glutathione reductase (FAD/NAD Rossmann motifs)**, **GSTs (typically ~200–250 aa domains)**, or **periplasmic γ-glutamyltransferase (signal peptide + Ntn processing)**), there’s **no strong evidence** that this protein directly uses **GSH as a substrate**.

So the best-supported answer from what we can see right now is: **likely NO (or at least not supported by sequence features alone)** ✅

If you want a crisp, defensible yes/no (e.g., for benchmarking), the “old-school reliable” route is:

1. Run **blastp** / **InterProScan** on the sequence to get a confident enzyme family/EC.
2. Then check whether that EC is known to take **GSH** as substrate (vs producing GSH, binding it, or not involving it at all).

If you paste the BLAST top hit (or predicted EC/domain hits), I can map it immediately to whether **GSH** is a bona fide substrate for that enzyme family.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)  
response = client.chat(r"Is the compound CC(C)(O)[C@@H](O)C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/3.fasta')


### GPT-5.2 Response

The SMILES **CC(C)(O)[C@@H](O)C(=O)[O-]** corresponds to a **dihydroxy branched-chain amino-acid intermediate** (chemically: a **C5H9O4− dihydroxy-methylbutyrate**; often discussed in the context of **branched-chain amino acid (Val/Ile) biosynthesis** 😊).

If this compound is a “true substrate” of an enzyme, the most common textbook case is **dihydroxy-acid dehydratase (IlvD family)**, which dehydrates dihydroxy-acids in the BCAA pathway.

Now, looking at the protein sequence in your uploaded **3.fasta** (≈ **397 aa**), it **does not show the hallmark IlvD [4Fe-4S] cysteine motif** (typically something like **C-(T/S)-X₂-C** in the catalytic region). Your sequence has very few cysteines overall and none matching that pattern, which makes an IlvD-like “dihydroxy-acid substrate” enzyme **unlikely**.

So, with the evidence we can extract from the FASTA alone: **probably NO** — this compound is **unlikely to be a substrate** of the enzyme in **3.fasta** ✅

If you want a rock-solid yes/no for benchmarking, the most reliable workflow is still the traditional one:

1. run **blastp / InterProScan** to identify the enzyme family (or EC),
2. then map that family to known substrates (if it’s IlvD-like → **yes**; if not → usually **no**).

If you paste the top BLAST hit / domain call, I can translate it into a substrate verdict immediately.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound NC(=O)C1=CN([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](N4C=NC5=C4N=CN=C5N)[C@H](OP(=O)([O-])[O-])[C@@H]3O)[C@@H](O)[C@H]2O)C=CC1 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/4.fasta')


### GPT-5.2 Response

That SMILES is **NAD⁺ (nicotinamide adenine dinucleotide, oxidized form)** — the canonical redox cofactor (and also a “substrate” for NAD-consuming enzymes like sirtuins / ADP-ribosyltransferases) 🧪

For the enzyme in your uploaded **4.fasta**, the sequence is ~**278 aa** and it **does not show the classic glycine-rich dinucleotide-binding (Rossmann) fingerprint** you’d expect for NAD(P)-dependent oxidoreductases (e.g., an N-terminal **VIGLGXGGXG / TGGXGGXG**-type motif). It also doesn’t resemble the usual NAD-consuming enzyme families by obvious motifs/architecture.

So, based on sequence features alone: **very likely NO — NAD⁺ is unlikely to be a true substrate/cofactor for the enzyme in this file.** ✅

If you want to lock this down for benchmarking (so the conclusion is defensible), the traditional reliable path is: **BLAST/InterPro → enzyme family/EC → check whether that family binds/consumes NAD⁺.** If you paste the top hit or predicted EC/domain, I can turn it into a firm substrate call immediately.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)  
response = client.chat(r"Is the compound O=C([O-])/C=C(/CCCC(=O)[O-])C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/5.fasta')


### GPT-5.2 Response

That SMILES **O=C([O-])/C=C(/CCCC(=O)[O-])C(=O)[O-]** is an **unsaturated tricarboxylate** (a “polycarboxylate + alkene” scaffold; metabolically it looks more like the kind of thing handled by **hydratases/dehydratases/lyases** in organic-acid pathways than by ATP-driven enzymes) 🧪

For the enzyme in your uploaded **5.fasta**: the sequence is **very large (~1235 aa)** and starts with a clear **P-loop NTPase / Walker A** motif (…**AAGAGTGKT**…). That architecture is typical of **ATP/GTP-driven enzymes (AAA+/P-loop families)**, not the classic **carboxylate-alkene processing enzymes** (e.g., aconitase/fumarase-class enzymes), which have very different conserved motifs and domain sizes.

So, based on the sequence features we can extract from **5.fasta** alone: **very likely NO — this compound is unlikely to be a substrate of that enzyme.** ✅

If you want to make this verdict “publication/benchmark-grade” (no handwaving): run a quick **BLAST/InterPro** on the 5.fasta sequence to nail the enzyme family/EC; once you have that, substrate mapping becomes deterministic. If you paste the top hit (or predicted EC/domain), I can translate it into a hard yes/no immediately.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound NC1=NC2=C(N=CN2[C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]2O)C(=O)N1 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/6.fasta')


### GPT-5.2 Response

That SMILES is **GTP (guanosine 5′-triphosphate)** 🧬⚡️

For the enzyme in your uploaded **6.fasta** (≈259 aa), the sequence **doesn’t show the usual “GTP-as-a-substrate” fingerprints**, especially the classic **P-loop/Walker A** nucleotide-binding motif (something like **[AGX₄GKST]**) that you see in many GTP/ATP-dependent enzymes (kinases, NTPases, many ligases). With that absence, **it’s unlikely this protein directly uses/consumes GTP as a substrate**.

So the best-supported call from sequence features alone is: **probably NO** ✅

If you want to make this airtight for benchmarking, the most reliable “traditional” next step is: run **BLAST/InterPro** on the 6.fasta sequence to identify the enzyme family/EC; once you know the family, GTP substrate usage becomes a straightforward yes/no mapping.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC(C)(O)[C@@H](O)C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/7.fasta')


### GPT-5.2 Response

The SMILES **CC(C)(O)[C@@H](O)C(=O)[O-]** is a branched-chain **dihydroxy-acid** (commonly discussed as **2,3-dihydroxy-3-methylbutyrate / 2,3-dihydroxyisovalerate–type** intermediate in Val/Ile biosynthesis) 🙂.

For the enzyme in your **7.fasta** (protein length ≈ **273 aa**), the sequence doesn’t look like the canonical enzyme families that take this kind of dihydroxy-acid as a substrate:

* It **doesn’t match IlvD/dihydroxy-acid dehydratase** (usually ~540–600 aa and carries the conserved **[4Fe-4S] Cys motif**, which your sequence essentially lacks).
* It also **doesn’t show a clear Rossmann NAD(P) motif** you’d expect if it were the reductoisomerase-type enzymes often adjacent in that pathway.

So the best-supported call from the sequence alone is: **likely NO**, this compound is **unlikely** to be a substrate of the enzyme in **7.fasta** ✅

If you want to make this verdict bulletproof (for a benchmark table), the “classic reliable” step is to annotate the protein family/EC (BLAST/InterPro), then substrate mapping becomes deterministic.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC(C)(O)[C@@H](O)C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/8.fasta')

### GPT-5.2 Response

That SMILES **CC(C)(O)[C@@H](O)C(=O)[O-]** is a branched-chain **dihydroxy-acid** (most commonly discussed as **2,3-dihydroxy-3-methylbutyrate / 2,3-dihydroxyisovalerate–type** intermediate in Val/Ile biosynthesis) 🧪

For the enzyme in your uploaded **8.fasta** (protein length ~**401 aa**), the sequence **does not** show the key fingerprint you’d expect for the canonical enzyme that uses this compound as a substrate, namely **IlvD / dihydroxy-acid dehydratase**:

* IlvD enzymes are typically ~**550–600 aa**
* and they carry the conserved **[4Fe-4S] cysteine motif** (often something like **C(T/S)X₂C**) required for catalysis
  Your 8.fasta sequence has **no such Cys motif**, making an IlvD-like substrate relationship very unlikely.

So, best-supported answer from sequence features alone: **likely NO**, this compound is **unlikely** to be a substrate of the enzyme in **8.fasta** ✅

If you want a “benchmark-grade” definitive call, the most reliable old-school workflow is still: **BLAST/InterPro → enzyme family/EC → substrate mapping**. If you paste the top hit/domain call, I’ll translate it into a hard yes/no immediately.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound NC1=NC2=C(N=CN2[C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]2O)C(=O)N1 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/9.fasta')

### GPT-5.2 Response

That SMILES is **GTP (guanosine 5′-triphosphate)** ⚡️🧬

Looking at the enzyme sequence in your **9.fasta** (≈239 aa), it **doesn’t contain the canonical GTP/ATP-binding fingerprints**, especially the **P-loop/Walker A motif** (typically something like **A/GxxxxGK[ST]**) that almost all enzymes using GTP as a chemical substrate rely on. I also don’t see other common small-GTPase/TRAFAC patterns.

So, based on the FASTA sequence alone: **very likely NO — GTP is unlikely to be a substrate of the enzyme in this file.** ✅

If you want to make this “benchmark-grade” certain, the traditional reliable path is still: **BLAST/InterPro → enzyme family/EC → substrate mapping** (then the yes/no becomes deterministic).


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound *[C@H](O)CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)([O-])OP(=O)([O-])OC[C@H]1O[C@@H](N2C=NC3=C2N=CN=C3N)[C@H](O)[C@@H]1OP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/10.fasta')

### GPT-5.2 Response

That SMILES is a **CoA thioester**, specifically a **3-hydroxyacyl-CoA** scaffold (the leading `*` means the acyl side chain is “unspecified/variable”), i.e., the kind of substrate you’d see in β-oxidation / acyl-CoA metabolism 🧪

For the enzyme in your **10.fasta** (sequence length ≈ **439 aa**), the primary-sequence “tells” for CoA-thioester enzymes aren’t there. In particular I don’t see the common conserved signatures you’d expect in the main families that actually **use 3-hydroxyacyl-CoA as substrate**, such as:

* **Enoyl-CoA hydratase / crotonase**-like enzymes (often have an **HGGX(10)D**-type motif),
* **3-hydroxyacyl-CoA dehydrogenases** (often have a **Rossmann** dinucleotide-binding motif),
* **thiolases** (have a characteristic catalytic **Cys** motif pattern),
* or obvious **CoA-binding** consensus patterns.

So, based on the FASTA sequence alone, the best-supported call is: **likely NO — this CoA thioester is unlikely to be a substrate of the enzyme in this file.** ✅

If you want this to be “benchmark-grade” certain (not motif-based inference), the reliable old-school path is: **BLAST/InterPro → enzyme family/EC → substrate mapping**. If you paste the top BLAST hit or predicted domains, I’ll translate it into a hard yes/no immediately.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound *O[C@H]1[C@@H](O)[C@H](*)O[C@@H]1COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/11.fasta')

### GPT-5.2 Response

That SMILES `*O[C@H]1[C@@H](O)[C@H](*)O[C@@H]1COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-]` is essentially a “ribose + 5′-triphosphate” core with two substituents left unspecified (`*`). In other words, it looks like a generic nucleotide-/activated-sugar–style scaffold rather than a fully specified small molecule.

For the enzyme in your uploaded **11.fasta**, the sequence is a large (~922 aa) eukaryotic-looking protein with long low-complexity regions and a central catalytic-looking domain that resembles a **protein kinase–like fold** (it contains a characteristic **…GLGK…**-type segment seen in kinase cores). Proteins of this type typically use **ATP** as the phosphate donor, and their “substrates” are usually **protein residues (Ser/Thr/Tyr)** rather than free ribose-triphosphate metabolites.

So, with the evidence available from the FASTA alone: **very likely NO** — this ribose-triphosphate scaffold is **unlikely** to be a true substrate of the enzyme in this file. ✅🙂

If you want to turn this into a rock-solid call for benchmarking, the conservative/old-school workflow is: annotate the protein family (BLAST/InterPro), then check whether it’s a bona fide kinase (ATP co-substrate; protein targets) versus a metabolic enzyme that truly consumes a ribose-triphosphate small molecule.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound *C(*)O a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/12.fasta')

### GPT-5.2 Response

That SMILES `*C(*)O` isn’t a fully specified compound 🙂 — the `*` atoms mean “some substituent I’m not defining”, so chemically it’s just “a generic alcohol center”. With that level of ambiguity, “is it a substrate?” isn’t well-posed in a strict enzymology sense (almost any enzyme class that acts on alcohol-containing molecules could match *some* instantiation of that pattern).

If I still force a practical call using your **12.fasta** sequence (≈253 aa): the protein does **not** look like a typical alcohol-oxidizing enzyme (e.g., no obvious N-terminal Rossmann fingerprint for NAD(P)-dependent dehydrogenases, and it doesn’t resemble the classic Zn-dependent ADH signatures either). So there’s **no sequence-based evidence** that “an alcohol like this” is a direct substrate.

Net: **likely NO (and the SMILES is too underspecified to claim YES reliably).** ✅


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound C=CC1=C(C)C2=[N+]3/C1=C\C1=C(C)C(CCC(=O)[O-])=C4C=C5C(CCC(=O)[O-])=C(C)C6=[N+]5[Fe-2]3(N41)N1/C(=C\\6)C(C)=C([C@@H](O)CC/C=C(\\C)CC/C=C(\\C)CCC=C(C)C)/C1=C/2 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/13.fasta')

### GPT-5.2 Response

The molecule you gave is a **heme-type porphyrin iron complex**, and based on the long hydrophobic isoprenoid-like tail plus two propionate groups, it’s most consistent with **heme a (or a heme a–like cofactor)**.

For the enzyme in your **13.fasta**: the protein is ~**218 aa** and contains a very clear **P-loop NTPase / Walker A** motif (something like **…GxxxxGKT…**, e.g., `…GPPGAGKGT…`). Proteins with this signature typically use **ATP or GTP as the chemical substrate** (hydrolysis / phosphate transfer), not bulky heme cofactors.

So the best-supported answer is: **very likely NO** — this heme-like compound is **not a substrate** of the enzyme in that file. ✅

If anything, heme a is usually a **bound cofactor** in respiratory complexes (e.g., cytochrome c oxidase), not something a small P-loop NTPase would directly convert.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)       
response = client.chat(r"Is the compound CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/14.fasta')

### GPT-5.2 Response

The SMILES you provided is **FMN (flavin mononucleotide; riboflavin-5′-phosphate)** 🧪

For the enzyme in your **14.fasta**: the sequence (~361 aa) is extremely **Ala/Val-rich** and, importantly, it **lacks the typical flavin-binding signatures** you’d expect if FMN were a real substrate/cofactor target (for example, the common glycine-rich FMN/FAD-binding patterns seen in flavoproteins are essentially absent; this sequence has almost no “GG”-type glycine clusters at all).

So, based on sequence evidence alone: **very likely NO — FMN is unlikely to be a substrate of the enzyme in this file.** ✅

If you want a “hard” call (benchmark-grade), the traditional reliable step is still: annotate the protein family/EC (BLAST/InterPro), then substrate mapping becomes deterministic (FMN-as-substrate mainly shows up in specific enzymes like FMN adenylyltransferase/FAD synthetase, flavin reductases, etc.).


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/15.fasta')

### GPT-5.2 Response

The compound
`CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-]`
is **FMN (flavin mononucleotide; riboflavin-5′-phosphate)** 🧪

For the enzyme in your **15.fasta** (protein length ≈ **465 aa**): its sequence is **very rich in Lys/Glu and looks largely helical/coiled-coil**, and it **does not show the common flavin-binding fingerprints** seen in enzymes that use/bind FMN (e.g., the typical glycine-rich Rossmann-like patterns or other conserved flavoprotein motifs).

So, based on sequence evidence alone: **very likely NO — FMN is unlikely to be a substrate of the enzyme in this file.** ✅

If you want a “benchmark-grade” definitive call: run **BLAST/InterPro** to identify the enzyme family/EC, then substrate mapping for FMN becomes straightforward.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/16.fasta')

### GPT-5.2 Response

The compound
`CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-]`
is **FMN (flavin mononucleotide; riboflavin-5′-phosphate)** 🧪

For the enzyme sequence in your **16.fasta** (≈ **663 aa**), the primary sequence **does not show the common flavin-binding fingerprints** typically seen in FMN/FAD-dependent enzymes (e.g., glycine-rich dinucleotide/flavin-binding patterns such as **GXGXXG** / Rossmann-like motifs, flavodoxin-like FMN loops, etc.). Based on that, **FMN is very unlikely to be a true substrate of this enzyme** ✅

Small nuance (so you can use this cleanly in evaluation): FMN is often a **bound cofactor** in specific flavoprotein families; when those motifs aren’t present, the safest call is **NO** unless an external annotation (BLAST/InterPro/EC) indicates a known FMN-dependent family.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC(C)(O)[C@@H](O)C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/17.fasta')

### GPT-5.2 Response

The SMILES **CC(C)(O)[C@@H](O)C(=O)[O-]** is most consistent with **2,3-dihydroxy-3-methylbutyrate (dihydroxyisovalerate)**, a classic intermediate in **branched-chain amino-acid (Val/Ile) biosynthesis** 🧪

For the enzyme in your **17.fasta** (≈ **445 aa**), the sequence strongly argues **against** it using this metabolite as a substrate:

* The best-known enzyme that *directly consumes* dihydroxyisovalerate is **IlvD (dihydroxy-acid dehydratase)**, which is usually **~550–600 aa** and requires an **[4Fe-4S] cluster**, reflected by a conserved **Cys motif** (and generally several cysteines).
* Your 17.fasta protein has **only one cysteine total**, and it **doesn’t show** the IlvD-type Fe-S/Cys signature.
* It also doesn’t look like the **IlvC** class (ketol-acid reductoisomerase), which would typically show a clear **Rossmann NAD(P) motif** near the N-terminus.

So the most defensible call from sequence features alone is: **likely NO — this compound is unlikely to be a substrate of the enzyme in 17.fasta.** ✅

If you want to “lock” the answer for evaluation: get a quick family/EC label (BLAST/InterPro), then we can map substrate scope deterministically.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound *[C@@H](O)CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)([O-])OP(=O)([O-])OC[C@H]1O[C@@H](N2C=NC3=C2N=CN=C3N)[C@H](O)[C@@H]1OP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/18.fasta')

### GPT-5.2 Response

The SMILES you gave is a **CoA thioester**, specifically a generic **3-hydroxyacyl-CoA** (the leading `*` means the acyl chain is unspecified) 🧪

For the enzyme in your **18.fasta**: the sequence is **only ~133 amino acids long** and it has **no recognizable CoA-binding / acyl-CoA enzyme motifs** (no HGG…D crotonase-type signature, no Rossmann NAD(P) motif, no catalytic Cys patterns typical of thiolases, etc.). Enzymes that truly use **3-hydroxyacyl-CoA** as a substrate are usually **much larger** and have very characteristic conserved features.

So the most defensible call from the sequence evidence is: **very likely NO — this 3-hydroxyacyl-CoA is not a substrate of the enzyme in 18.fasta.** ✅


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC(C)(O)[C@@H](O)C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/19.fasta')

### GPT-5.2 Response

The compound **CC(C)(O)[C@@H](O)C(=O)[O-]** is a branched-chain **dihydroxy-acid** (most commonly **2,3-dihydroxy-3-methylbutyrate / dihydroxyisovalerate**, a Val/Ile-biosynthesis intermediate) 🧪

For the enzyme in your **19.fasta** (≈ **261 aa**), the sequence does **not** match the canonical enzyme families that use this metabolite as a true substrate:

* It’s **far too small** and lacks the conserved **[4Fe-4S] Cys motif** typical of **IlvD (dihydroxy-acid dehydratase)**, the “textbook” consumer of this substrate.
* It also lacks the common **Rossmann/NAD(P)-binding fingerprints** you’d expect if it were an **IlvC-type** reductoisomerase-related enzyme.

So the most defensible call from the sequence evidence is: **likely NO — this compound is unlikely to be a substrate of the enzyme in 19.fasta.** ✅


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound *O[C@H]1[C@@H](O)[C@H](*)O[C@@H]1COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])OC[C@H]1O[C@@H](N2C=[N+](C)C3=C2N=C(N)NC3=O)[C@H](O)[C@@H]1O a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/20.fasta')

### GPT-5.2 Response

The compound you wrote, `*O[C@H]1…OP(=O)…OP(=O)…OP(=O)…OC[C@H]1…`, is essentially a **5′ mRNA cap–like dinucleotide**: **m⁷GpppN** (7-methylguanosine linked via a 5′-5′ triphosphate to another nucleoside; the `*` means that second nucleoside/base is unspecified) 🧬

For the enzyme in your **20.fasta**, the sequence shows a very clear **P-loop NTPase + “DEAD” motif** (Walker A `…TGSGKT…` and `…VLDEAD…`), which is characteristic of a **DEAD-box RNA helicase** family protein. These enzymes use **ATP** (and RNA) as their functional substrates: they bind RNA and hydrolyze ATP to remodel RNA/protein complexes.

So the most defensible answer is: **very likely NO** ✅
A free **cap dinucleotide (m⁷GpppN)** is not the typical “substrate” for a DEAD-box helicase. (The cap can be part of capped RNA, but that’s a different statement than the enzyme using the isolated cap molecule as a substrate.)

If you want, I can also tell you what motifs/domains in the 20.fasta support the DEAD-box call, so you can cite it cleanly in your benchmark notes 🙂


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound NC(=O)C1=CC=C[N+]([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](N4C=NC5=C4N=CN=C5N)[C@H](O)[C@@H]3O)[C@@H](O)[C@H]2O)=C1 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/21.fasta')

### GPT-5.2 Response

The compound
`NC(=O)C1=CC=C[N+]([C@@H]2O…)=C1`
is **NAD⁺ (nicotinamide adenine dinucleotide, oxidized form)** 🧪

For the enzyme in your **21.fasta**: the protein is small (~**212 aa**) and its N-terminus **does not show a Rossmann-type dinucleotide-binding motif** (the classic **VIGLGXGGXG / TGGXGGXG**-style glycine-rich fingerprint) that you typically see in NAD(P)-dependent oxidoreductases. I also don’t see other obvious sequence signatures suggesting an NAD-consuming enzyme family.

So, based on the sequence evidence available from the FASTA alone: **very likely NO — NAD⁺ is unlikely to be a true substrate/cofactor for this enzyme.** ✅

If you want this to be absolutely “benchmark-grade” certain, the old reliable path is: **BLAST/InterPro → enzyme family/EC → confirm whether that family binds or consumes NAD⁺**.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound [H][C@@]12CC[C@]([H])([C@H](C)CCCC(C)C)[C@@]1(C)CCC/C2=C\C=C1\C[C@@H](O)CCC1=C a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/22.fasta')

### GPT-5.2 Response

The SMILES you gave is a **hydrophobic sterol/secosteroid-like alcohol** (cholesterol / vitamin-D–like scaffold: lots of fused rings + one OH, no obvious hydrolysable bond) 🧪

The protein in **22.fasta** looks like a **secreted serine hydrolase**: it has an N-terminal **signal peptide** (secretory leader) and a very clear **serine-active-site “GXSXG” motif** (`…IGSGGS…`). Enzymes with this architecture typically act on **peptide/ester substrates** (proteases/esterases/lipases), where there’s a **cleavable amide/ester bond**.

Your compound is basically a **neutral steroid alcohol** with **no ester/amide bond to hydrolyze**, so it’s **very unlikely to be a substrate** of the enzyme in this file. ✅

If you want to make the call “benchmark-grade” (zero guesswork), run a quick BLAST/InterPro on the 22.fasta sequence to confirm the exact family (protease vs esterase/lipase), then substrate scope becomes deterministic.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound [1*]CCC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)([O-])OP(=O)([O-])OC[C@H]1O[C@@H](N2C=NC3=C2N=CN=C3N)[C@H](O)[C@@H]1OP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/23.fasta')

### GPT-5.2 Response

The compound
`[1*]CCC(=O)S…` is an **acyl-CoA thioester** (a CoA “activated fatty acid”/acyl group; the `[1*]` means the acyl chain is not fully specified) 🧪

For the enzyme in your uploaded **23.fasta**: the protein is **~1464 aa** long and contains a clear **P-loop NTPase / Walker A** motif around the N-terminal region (e.g., `…VAKSSGKT…`). It **does not** show the typical sequence hallmarks of enzymes that *directly use acyl-CoA thioesters as substrates* (e.g., crotonase/enoyl-CoA hydratase motifs like **HGG…D**, thiolase catalytic **Cys** patterns, obvious CoA-transferase signatures, etc.).

So the most defensible call from the sequence features is: **very likely NO** ✅
(i.e., this acyl-CoA is unlikely to be a true substrate of the enzyme in this file.)

If you want a “benchmark-grade” definitive label: a quick **BLAST/InterPro** family assignment for the 23.fasta protein will make the substrate decision essentially deterministic.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound NC(=O)C1=CC=C[N+]([C@@H]2O[C@H](COP(=O)([O-])OP(=O)([O-])OC[C@H]3O[C@@H](N4C=NC5=C4N=CN=C5N)[C@H](O)[C@@H]3O)[C@@H](O)[C@H]2O)=C1 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/24.fasta')

### GPT-5.2 Response

That molecule is **NAD⁺ (nicotinamide adenine dinucleotide, oxidized form)** 🧪

For the protein in your **24.fasta**, the sequence is **very small (~153 aa)** and it **does not contain a recognizable NAD-binding “Rossmann” fingerprint** (the classic N-terminus glycine-rich pattern like **VIGLGXGGXG / TGGXGGXG**) that you typically see in NAD(P)-dependent oxidoreductases. It also doesn’t resemble the usual **NAD-consuming** enzyme families (e.g., sirtuins, ADP-ribosyltransferases, NADases), which have distinct conserved motifs and are generally larger.

So the most defensible call from the sequence evidence is: **very likely NO — NAD⁺ is unlikely to be a substrate (or required cofactor) for the enzyme in this file.** ✅


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CCC1CCC2C3CCC4CCCCC4(C)C3CCC12C a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/25.fasta')

### GPT-5.2 Response

The compound `CCC1CCC2C3CCC4CCCCC4(C)C3CCC12C` is a **fully hydrophobic steroid/sterane-like hydrocarbon** (no heteroatoms, no ester/amide/phosphate, basically no obvious “handle” for enzyme chemistry) 🧪

The protein in your **25.fasta** starts with a very clear **P-loop NTPase (Walker A) motif**: `…VIVGETGCGKST…`, and the overall size (~703 aa) is consistent with an **ATP/GTP-driven enzyme** (often a transporter-associated ATPase or another NTPase family). Those proteins’ real small-molecule substrates are typically **ATP (sometimes GTP)**, not inert sterane hydrocarbons.

So the most defensible call is: **very likely NO** ✅
This sterane-like hydrocarbon is extremely unlikely to be a substrate of the enzyme in that file.


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC1=CC2=C(C=C1C)N(C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])OP(=O)([O-])OC[C@H]1O[C@@H](N3C=NC4=C3N=CN=C4N)[C@H](O)[C@@H]1O)C1=NC(=O)[N-]C(=O)C1=N2 a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/26.fasta')

### GPT-5.2 Response

That SMILES is **FAD (flavin adenine dinucleotide)** — a flavin redox **cofactor** (often tightly bound in flavoproteins), not usually a freely turned-over “metabolic substrate” 🧪

For the enzyme in your uploaded **26.fasta**: the sequence (~520 aa) is **highly hydrophobic with many predicted transmembrane helices** and **lacks the classic FAD/FMN-binding fingerprints** you’d expect in flavoproteins (e.g., Rossmann-like glycine-rich motifs). Proteins with this architecture are typically **membrane receptors/transport-related proteins**, not soluble flavin-dependent oxidoreductases.

So the most defensible call is: **very likely NO — FAD is not a substrate of the protein in this file.** ✅


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound *[C@@H]1O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/27.fasta')

### GPT-5.2 Response

Based on the FASTA sequence, the enzyme in the file is very likely a **eukaryotic serine/threonine protein kinase**. The sequence contains hallmark kinase motifs, including:

* The **VAIK** catalytic lysine motif
* The **HRD** motif in the catalytic loop
* The **DFG** motif for Mg²⁺ coordination
* A conserved **ATP-binding glycine-rich region**

The C-terminal region is proline-rich and regulatory, which is typical of many human protein kinases.

Protein kinases catalyze:

• Transfer of a phosphate group
• From **ATP**
• To a serine/threonine residue on a protein substrate

Now, regarding your compound:

`[C@@H]1O[C@H](COP(=O)([O-])OP(=O)([O-])OP(=O)([O-])[O-])[C@@H](O)[C@H]1O`

This structure corresponds to a **ribose triphosphate lacking a nucleobase**. It resembles the sugar–triphosphate portion of ATP but does **not** contain adenine.

Kinases specifically recognize **ATP**, not just a ribose-triphosphate fragment. The adenine base is essential for binding in the ATP pocket via:

• π–π stacking
• Hydrogen bonding to hinge residues
• Specific orientation within the nucleotide-binding cleft

Without the adenine moiety, binding affinity would be negligible, and catalysis would not occur.

Therefore:

**No — this compound is not a substrate of the enzyme in the file.**

The enzyme’s substrates are:

1. **ATP** (as phosphate donor)
2. **Protein substrates** containing Ser/Thr residues

Your compound lacks the nucleobase required for kinase recognition and thus would not function as a substrate.

If you’d like, I can also map the ATP-binding residues onto the sequence to make this even more explicit 🔬✨


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound O=C([O-])CCC(=O)C(=O)[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/28.fasta')

### GPT-5.2 Response

The compound

**O=C([O-])CCC(=O)C(=O)[O-]**

is **2-oxoglutarate (α-ketoglutarate)**, a central TCA cycle intermediate and a very common metabolic substrate.

From the FASTA sequence in the uploaded file, the enzyme contains:

• A conserved **PLP-binding lysine** motif
• Sequence similarity to class I aminotransferases
• The characteristic aminotransferase fold (Asp/Glu-rich active-site region)

These features indicate the enzyme is most consistent with a **PLP-dependent aminotransferase**.

Aminotransferases typically catalyze:

amino acid + α-ketoglutarate ⇌ α-keto acid + glutamate

In this reaction, **α-ketoglutarate is a canonical substrate**.

Therefore:

**Yes — O=C([O-])CCC(=O)C(=O)[O-] (α-ketoglutarate) is very likely a substrate of the enzyme in this file.**

If you want, I can also predict the most probable amino acid partner based on sequence homology and active-site residues 🔬✨


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound [NH3+]CCO a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/29.fasta')

### GPT-5.2 Response

The compound **[NH3+]CCO** is **ethanolamine**, a small primary amino alcohol.

From inspection of the FASTA sequence in this file, the enzyme:

• Lacks PLP-binding motifs (no conserved Lys for Schiff base formation)
• Does not show aminotransferase or decarboxylase signatures
• Contains Rossmann-like nucleotide-binding features
• Is consistent with a dehydrogenase/oxidoreductase family rather than a small amine-processing enzyme

Ethanolamine is typically a substrate for:

• Ethanolamine ammonia-lyase
• Ethanolamine kinase
• Ethanolamine oxidase

Those enzymes have very specific catalytic machinery for handling primary amines or phosphoethanolamine formation.

The enzyme in your file does **not** display sequence features consistent with known ethanolamine-metabolizing enzymes.

Therefore:

**No — [NH3+]CCO (ethanolamine) is very unlikely to be a substrate of the enzyme in this file.**

If you'd like, I can also classify this enzyme more precisely and suggest its most probable metabolic substrate class 🔬✨


In [ ]:
client = GPTChatClient(system_prompt=system_prompt)   
response = client.chat(r"Is the compound CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-] a substrate of the enzyme in this file?", fasta_file='./data/substrate_determination/30.fasta')

### GPT-5.2 Response

The compound

**CC1=C(C)C=C2C(=C1)NC1=C(NC(=O)NC1=O)N2C[C@H](O)[C@H](O)[C@H](O)COP(=O)([O-])[O-]**

is a **flavin mononucleotide (FMN) derivative** — specifically riboflavin-5′-phosphate with methyl substitutions on the isoalloxazine ring.

Now looking at the FASTA sequence in this file:

The enzyme contains:

• A classical **Rossmann-like fold motif (GXGXXG)**
• Conserved residues typical of NAD/FAD-binding oxidoreductases
• Length (~180–250 aa range) consistent with flavin-dependent enzymes

However — and this is the key point — flavin-dependent enzymes almost always use:

• **FMN or FAD as tightly bound cofactors**, not substrates
• The flavin moiety is typically prosthetic (bound in the active site)

The structure you provided corresponds to **FMN itself**, not a reduced substrate to be chemically transformed.

Unless the enzyme is specifically a:

• FMN phosphatase
• Riboflavin kinase
• FAD synthetase

FMN is generally **not a catalytic substrate**, but rather a cofactor.

The sequence in your file does not show signatures of:

• Riboflavin kinase (RFK motif signatures absent)
• FAD synthetase (no large bifunctional architecture)
• Known FMN-processing enzymes

Instead, it resembles a **flavin-dependent oxidoreductase**, meaning FMN would bind as a cofactor.

Therefore:

**No — this compound (FMN) is not a substrate of the enzyme in this file.**
It would, at most, function as a bound cofactor if compatible.

If you'd like, I can further classify the enzyme family from the sequence and determine whether it is FMN- or FAD-dependent 🔬✨
